In [1]:
import sys
sys.path.append("..")
import pandas as pd

from torch.utils.data import DataLoader
from model_zoo import get_model
from dataset_zoo import VG_Relation, VG_Attribution

ModuleNotFoundError: No module named 'model_zoo'

In [2]:
# !pip install easydict
# !pip install gdown
# !sudo apt-get install unzip

In [3]:
# Please put your data root directory below. We'll download VG-Relation and VG-Attribution images here. 
# Will be a 1GB zip file (a subset of GQA).
root_dir="~/.cache" 


In [4]:
import os
os.environ["LLM2VEC_VERSION"] = "3.1_latent_mixd15m"
model, preprocess = get_model(model_name="llm2clip:ViT-L/14@336px", device="cuda:0", root_dir=root_dir,pretrained='/blob/hwq/data/tune_logs/T_vitEVA02-CLIP-L-14_32x8*16_lr1e-5_Rd30m_3.1_latent_mixd15m_eval_4ep-2025_01_08-20/checkpoints/epoch_4/mp_rank_00_model_states.pt')


/home/aiscuser/waq/instructCLIP/vision-language-models-are-bows/model_zoo/llm2clip_models.py:55: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast()


3.1_latent_mixd15m


You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/home/aiscuser/waq/instructCLIP/fusemix/EVA-CLIP/rei/eva_clip/factory.py:90: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_locat

text.text.model.embed_tokens.weight torch.bfloat16
text.text.model.layers.0.self_attn.q_proj.weight torch.bfloat16
text.text.model.layers.0.self_attn.k_proj.weight torch.bfloat16
text.text.model.layers.0.self_attn.v_proj.weight torch.bfloat16
text.text.model.layers.0.self_attn.o_proj.weight torch.bfloat16
text.text.model.layers.0.mlp.gate_proj.weight torch.bfloat16
text.text.model.layers.0.mlp.up_proj.weight torch.bfloat16
text.text.model.layers.0.mlp.down_proj.weight torch.bfloat16
text.text.model.layers.0.input_layernorm.weight torch.bfloat16
text.text.model.layers.0.post_attention_layernorm.weight torch.bfloat16
text.text.model.layers.1.self_attn.q_proj.weight torch.bfloat16
text.text.model.layers.1.self_attn.k_proj.weight torch.bfloat16
text.text.model.layers.1.self_attn.v_proj.weight torch.bfloat16
text.text.model.layers.1.self_attn.o_proj.weight torch.bfloat16
text.text.model.layers.1.mlp.gate_proj.weight torch.bfloat16
text.text.model.layers.1.mlp.up_proj.weight torch.bfloat16
t

/home/aiscuser/waq/instructCLIP/vision-language-models-are-bows/model_zoo/__init__.py:113: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(pretrained)


In [8]:
for name,param in model.model.named_parameters():
    print(name,param.dtype)  # 检查每个参数的数据类型


In [6]:
# !pip install 
!nvidia-

In [9]:
# Get the VG-R dataset
vgr_dataset = VG_Relation(image_preprocess=preprocess, download=True, root_dir=root_dir)
vgr_loader = DataLoader(vgr_dataset, batch_size=64, shuffle=False, num_workers=16)

# Compute the scores for each test case
vgr_scores = model.get_retrieval_scores_batched(vgr_loader)


Computing retrieval scores:   0%|                                           | 0/375 [00:00<?, ?it/s]

/home/aiscuser/miniconda3/envs/fusemix/lib/python3.8/site-packages/apex/_autocast_utils.py:26: FutureWarning: `torch.cuda.amp.autocast_mode._cast(value, dtype)` is deprecated. Please use `torch.amp.autocast_mode._cast(value, 'cuda', dtype)` instead.
  return torch.cuda.amp.autocast_mode._cast(args, torch.get_autocast_gpu_dtype())
We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)
The input hidden states seems to be silently casted in float32, this might be related to the fact you have upcasted embedding or layer norm layers in float32. We will cast back the input in torch.float16.
/home/aiscuser/miniconda3/envs/fusemix/lib/python3.8/contextlib.py:83: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn

In [10]:
# Evaluate the macro accuracy
vgr_records = vgr_dataset.evaluate_scores(vgr_scores)
symmetric = ['adjusting', 'attached to', 'between', 'bigger than', 'biting', 'boarding', 'brushing', 'chewing', 'cleaning', 'climbing', 'close to', 'coming from', 'coming out of', 'contain', 'crossing', 'dragging', 'draped over', 'drinking', 'drinking from', 'driving', 'driving down', 'driving on', 'eating from', 'eating in', 'enclosing', 'exiting', 'facing', 'filled with', 'floating in', 'floating on', 'flying', 'flying above', 'flying in', 'flying over', 'flying through', 'full of', 'going down', 'going into', 'going through', 'grazing in', 'growing in', 'growing on', 'guiding', 'hanging from', 'hanging in', 'hanging off', 'hanging over', 'higher than', 'holding onto', 'hugging', 'in between', 'jumping off', 'jumping on', 'jumping over', 'kept in', 'larger than', 'leading', 'leaning over', 'leaving', 'licking', 'longer than', 'looking in', 'looking into', 'looking out', 'looking over', 'looking through', 'lying next to', 'lying on top of', 'making', 'mixed with', 'mounted on', 'moving', 'on the back of', 'on the edge of', 'on the front of', 'on the other side of', 'opening', 'painted on', 'parked at', 'parked beside', 'parked by', 'parked in', 'parked in front of', 'parked near', 'parked next to', 'perched on', 'petting', 'piled on', 'playing', 'playing in', 'playing on', 'playing with', 'pouring', 'reaching for', 'reading', 'reflected on', 'riding on', 'running in', 'running on', 'running through', 'seen through', 'sitting behind', 'sitting beside', 'sitting by', 'sitting in front of', 'sitting near', 'sitting next to', 'sitting under', 'skiing down', 'skiing on', 'sleeping in', 'sleeping on', 'smiling at', 'sniffing', 'splashing', 'sprinkled on', 'stacked on', 'standing against', 'standing around', 'standing behind', 'standing beside', 'standing in front of', 'standing near', 'standing next to', 'staring at', 'stuck in', 'surrounding', 'swimming in', 'swinging', 'talking to', 'topped with', 'touching', 'traveling down', 'traveling on', 'tying', 'typing on', 'underneath', 'wading in', 'waiting for', 'walking across', 'walking by', 'walking down', 'walking next to', 'walking through', 'working in', 'working on', 'worn on', 'wrapped around', 'wrapped in', 'by', 'of', 'near', 'next to', 'with', 'beside', 'on the side of', 'around']
df = pd.DataFrame(vgr_records)
df = df[~df.Relation.isin(symmetric)]
print(f"VG-Relation Macro Accuracy: {df.Accuracy.mean()}")

VG-Relation Macro Accuracy: 0.5368894748523513


In [8]:
# Get the VG-A dataset
vga_dataset = VG_Attribution(image_preprocess=preprocess, download=True, root_dir=root_dir)
vga_loader = DataLoader(vga_dataset, batch_size=16, shuffle=False)
# Compute the scores for each test case
vga_scores = model.get_retrieval_scores_batched(vga_loader)


Computing retrieval scores:   0%|                                       | 0/1797 [00:00<?, ?it/s]

Computing retrieval scores:   2%|▋                             | 40/1797 [00:14<10:58,  2.67it/s]


KeyboardInterrupt: 

In [ ]:
# Evaluate the macro accuracy
vga_records = vga_dataset.evaluate_scores(vga_scores)
df = pd.DataFrame(vga_records)
print(f"VG-Attribution Macro Accuracy: {df.Accuracy.mean()}")